# 🌞 Clustering Solar Energy Production Zones
## Spinnaker Analytics — Data Science Internship Project
---
**Author:** Shijan Shiby  
**Organization:** Spinnaker Analytics  
**Project Domain:** Data Science & Artificial Intelligence  
**Objective:** Cluster geographical areas (counties) in New York State based on their solar energy production characteristics to support strategic targeting, project feasibility, and data-driven decision making.

---

## 📋 Table of Contents
1. [Setup & Imports](#1-setup)
2. [Data Loading & Initial Exploration](#2-eda)
3. [Data Cleaning & Preprocessing](#3-cleaning)
4. [Feature Engineering](#4-features)
5. [Exploratory Data Analysis (EDA)](#5-plots)
6. [Clustering — Algorithm Selection](#6-selection)
7. [K-Means Clustering](#7-kmeans)
8. [Hierarchical Clustering](#8-hierarchical)
9. [DBSCAN Clustering](#9-dbscan)
10. [Algorithm Comparison & Best Model](#10-comparison)
11. [Cluster Profiling & Insights](#11-profiling)
12. [Interactive Map Visualization](#12-map)
13. [Business Recommendations](#13-recommendations)


## 1. Setup & Imports <a id='1-setup'></a>

In [ ]:
# ── Core libraries ───────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Machine Learning ──────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from scipy.cluster.hierarchy import dendrogram, linkage

# ── Visualization ──────────────────────────────────────────────────────────
import plotly.express as px
import plotly.graph_objects as go

# ── Settings ──────────────────────────────────────────────────────────────
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:,.2f}'.format)

print('✅ All libraries loaded successfully')
print(f'   pandas {pd.__version__} | numpy {np.__version__}')

## 2. Data Loading & Initial Exploration <a id='2-eda'></a>

In [ ]:
# Load the dataset
df = pd.read_csv('Statewide_Solar_Projects.csv', low_memory=False)

print('=' * 60)
print(f'  Dataset Shape : {df.shape[0]:,} rows × {df.shape[1]} columns')
print('=' * 60)
df.head(5)

In [ ]:
# Data types and missing values overview
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
summary = pd.DataFrame({'dtype': df.dtypes, 'missing': missing, 'missing_%': missing_pct})
print(summary.to_string())

In [ ]:
# Basic statistical summary
df.describe()

## 3. Data Cleaning & Preprocessing <a id='3-cleaning'></a>

In [ ]:
# ── Fill missing categorical values ─────────────────────────────────────
df['Developer'].fillna('Unknown', inplace=True)
df['City/Town'].fillna('Unknown', inplace=True)

# ── Drop Energy Storage — 98% missing, not useful for clustering ─────────
df.drop(columns=['Energy Storage System Size (kWac)'], inplace=True)

# ── Remove rows with missing Project ID (only 3 rows) ────────────────────
df.dropna(subset=['Project ID'], inplace=True)

# ── Convert dates ─────────────────────────────────────────────────────────
df['Interconnection Date'] = pd.to_datetime(df['Interconnection Date'], errors='coerce')
df['Data Through Date']    = pd.to_datetime(df['Data Through Date'],    errors='coerce')

# ── Remove outliers in energy production (top 0.1%) ──────────────────────
energy_col = 'Estimated Annual PV Energy Production (kWh)'
upper = df[energy_col].quantile(0.999)
df = df[df[energy_col] <= upper]

print(f'✅ Clean dataset shape: {df.shape}')
print(f'   Date range: {df["Interconnection Date"].min().date()} → {df["Interconnection Date"].max().date()}')
print(f'   Counties  : {df["County"].nunique()}')
print(f'   Utilities : {df["Utility"].nunique()} — {df["Utility"].unique().tolist()}')

## 4. Feature Engineering <a id='4-features'></a>
> We aggregate raw project-level data to **county level**, creating meaningful features for clustering.

In [ ]:
# ── County-level aggregation ─────────────────────────────────────────────
energy_col = 'Estimated Annual PV Energy Production (kWh)'

county = df.groupby('County').agg(
    total_projects        = ('Project ID', 'count'),
    total_energy_kwh      = (energy_col, 'sum'),
    avg_energy_kwh        = (energy_col, 'mean'),
    median_energy_kwh     = (energy_col, 'median'),
    std_energy_kwh        = (energy_col, 'std'),
    avg_system_size_kw    = ('Estimated PV System Size (kWdc)', 'mean'),
    total_capacity_kw     = ('Estimated PV System Size (kWdc)', 'sum'),
    avg_kwac              = ('PV System Size (kWac)', 'mean'),
    unique_developers     = ('Developer', 'nunique'),
    unique_utilities      = ('Utility', 'nunique'),
).reset_index()

# ── Derived features ──────────────────────────────────────────────────────
county['energy_per_project']   = county['total_energy_kwh']  / county['total_projects']
county['capacity_per_project'] = county['total_capacity_kw'] / county['total_projects']
county['total_energy_gwh']     = county['total_energy_kwh']  / 1_000_000
county['kw_efficiency']        = county['avg_energy_kwh']    / county['avg_system_size_kw']

print(f'County feature matrix: {county.shape}')
county.head()

## 5. Exploratory Data Analysis <a id='5-plots'></a>

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle('Solar Energy Production — Exploratory Data Analysis\nNew York State', fontsize=18, fontweight='bold', y=0.98)

# Top 15 counties
top15 = county.nlargest(15, 'total_energy_gwh')
axes[0,0].barh(top15['County'], top15['total_energy_gwh'],
               color=plt.cm.YlOrRd(np.linspace(0.4,0.9,15)))
axes[0,0].set_xlabel('Total Energy Production (GWh)')
axes[0,0].set_title('Top 15 Counties by Total Solar Energy Production', fontweight='bold')
axes[0,0].invert_yaxis()

# Projects vs Energy
sc = axes[0,1].scatter(county['total_projects'], county['total_energy_gwh'],
                        c=county['avg_system_size_kw'], cmap='viridis', s=80, alpha=0.8)
plt.colorbar(sc, ax=axes[0,1], label='Avg System Size (kWdc)')
axes[0,1].set_xlabel('Number of Projects')
axes[0,1].set_ylabel('Total Energy (GWh)')
axes[0,1].set_title('Projects vs Total Energy by County', fontweight='bold')
for _, row in county.nlargest(6,'total_energy_gwh').iterrows():
    axes[0,1].annotate(row['County'], (row['total_projects'], row['total_energy_gwh']), fontsize=8)

# Distribution
axes[1,0].hist(county['avg_energy_kwh']/1000, bins=20, color='steelblue', edgecolor='white', alpha=0.85)
axes[1,0].set_xlabel('Avg Energy per Project (MWh)')
axes[1,0].set_ylabel('Number of Counties')
axes[1,0].set_title('Distribution of Avg Energy per Project', fontweight='bold')

# Utility pie
util_counts = df['Utility'].value_counts()
axes[1,1].pie(util_counts, labels=util_counts.index, autopct='%1.1f%%',
              colors=plt.cm.Set3(np.linspace(0,1,len(util_counts))), startangle=140)
axes[1,1].set_title('Project Distribution by Utility', fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Clustering — Algorithm Selection & Optimal K <a id='6-selection'></a>

In [ ]:
# ── Feature scaling ──────────────────────────────────────────────────────
FEATURES = ['total_energy_kwh', 'avg_energy_kwh', 'total_projects',
            'avg_system_size_kw', 'total_capacity_kw',
            'energy_per_project', 'capacity_per_project']

X = county[FEATURES].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('✅ Features scaled with StandardScaler')
print(f'   Feature matrix: {X_scaled.shape}')

In [ ]:
# ── Elbow Method + Silhouette Score ──────────────────────────────────────
inertias, sil_scores, k_range = [], [], range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Optimal Number of Clusters — K-Means Evaluation', fontsize=14, fontweight='bold')

axes[0].plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(x=4, color='red', linestyle='--', alpha=0.7, label='Selected k=4')
axes[0].set_xlabel('Number of Clusters (k)'); axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method', fontweight='bold'); axes[0].legend(); axes[0].grid(alpha=0.3)

best_k = list(k_range)[np.argmax(sil_scores)]
axes[1].plot(k_range, sil_scores, 'gs-', linewidth=2, markersize=8)
axes[1].axvline(x=best_k, color='red', linestyle='--', alpha=0.7, label=f'Best k={best_k}')
axes[1].set_xlabel('Number of Clusters (k)'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score', fontweight='bold'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()
print(f'Best k by Silhouette Score: {best_k}')
print('We select k=4 for business interpretability (4 distinct strategic zones)')

## 7. K-Means Clustering <a id='7-kmeans'></a>

In [ ]:
# ── Fit K-Means with k=4 ─────────────────────────────────────────────────
km_final = KMeans(n_clusters=4, random_state=42, n_init=10)
county['kmeans_cluster'] = km_final.fit_predict(X_scaled)

# Re-label clusters 0→3 by total energy (0=lowest, 3=highest)
cluster_energy = county.groupby('kmeans_cluster')['total_energy_gwh'].sum().sort_values()
remap = {old: new for new, old in enumerate(cluster_energy.index)}
county['cluster'] = county['kmeans_cluster'].map(remap)

ZONE_NAMES  = {0:'Zone D — Emerging', 1:'Zone C — Moderate',
               2:'Zone B — High Volume', 3:'Zone A — Premium'}
ZONE_COLORS = {0:'#95a5a6', 1:'#f39c12', 2:'#3498db', 3:'#e74c3c'}
county['zone_label'] = county['cluster'].map(ZONE_NAMES)

km_sil = silhouette_score(X_scaled, county['cluster'])
km_db  = davies_bouldin_score(X_scaled, county['cluster'])
km_ch  = calinski_harabasz_score(X_scaled, county['cluster'])

print(f'K-Means (k=4) Metrics:')
print(f'  Silhouette Score      : {km_sil:.4f}  (higher = better, max 1.0)')
print(f'  Davies-Bouldin Score  : {km_db:.4f}  (lower  = better)')
print(f'  Calinski-Harabasz     : {km_ch:.2f} (higher = better)')

In [ ]:
# ── Visualize K-Means results ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('K-Means Clustering — Solar Energy Production Zones', fontsize=15, fontweight='bold')

for c in range(4):
    mask = county['cluster'] == c
    axes[0].scatter(county.loc[mask,'total_projects'],
                    county.loc[mask,'total_energy_gwh'],
                    color=ZONE_COLORS[c], label=ZONE_NAMES[c], s=100, alpha=0.85, edgecolors='white')
    for _, row in county[mask].iterrows():
        if row['total_energy_gwh'] > 50:
            axes[0].annotate(row['County'], (row['total_projects'], row['total_energy_gwh']), fontsize=8)

axes[0].set_xlabel('Total Projects'); axes[0].set_ylabel('Total Energy (GWh)')
axes[0].set_title('Energy vs Projects — Clusters', fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)

cluster_totals = county.groupby('cluster')['total_energy_gwh'].sum()
bars = axes[1].bar([ZONE_NAMES[c] for c in cluster_totals.index],
                   cluster_totals.values,
                   color=[ZONE_COLORS[c] for c in cluster_totals.index],
                   edgecolor='white', width=0.6)
for bar, val in zip(bars, cluster_totals.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
                 f'{val:.0f} GWh', ha='center', fontweight='bold')
axes[1].set_ylabel('Total Energy (GWh)'); axes[1].set_title('Total Energy per Zone', fontweight='bold')
axes[1].grid(axis='y', alpha=0.3); plt.xticks(rotation=15, ha='right')
plt.tight_layout(); plt.show()

## 8. Hierarchical Clustering <a id='8-hierarchical'></a>

In [ ]:
# ── Dendrogram ───────────────────────────────────────────────────────────
Z = linkage(X_scaled, method='ward')
fig, ax = plt.subplots(figsize=(16, 6))
dendrogram(Z, labels=county['County'].values, ax=ax, color_threshold=6,
           leaf_rotation=90, leaf_font_size=8)
ax.axhline(y=6, color='red', linestyle='--', alpha=0.7, label='Cut = 4 clusters')
ax.set_title('Hierarchical Clustering Dendrogram — NY Counties', fontsize=14, fontweight='bold')
ax.set_ylabel('Ward Distance'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── Fit Agglomerative Clustering ─────────────────────────────────────────
hc = AgglomerativeClustering(n_clusters=4, linkage='ward')
county['hc_cluster'] = hc.fit_predict(X_scaled)

hc_sil = silhouette_score(X_scaled, county['hc_cluster'])
hc_db  = davies_bouldin_score(X_scaled, county['hc_cluster'])
hc_ch  = calinski_harabasz_score(X_scaled, county['hc_cluster'])
print(f'Hierarchical (k=4) Metrics:')
print(f'  Silhouette Score     : {hc_sil:.4f}')
print(f'  Davies-Bouldin Score : {hc_db:.4f}')
print(f'  Calinski-Harabasz    : {hc_ch:.2f}')

## 9. DBSCAN Clustering <a id='9-dbscan'></a>

In [ ]:
# ── DBSCAN ───────────────────────────────────────────────────────────────
db = DBSCAN(eps=1.5, min_samples=3)
county['dbscan_cluster'] = db.fit_predict(X_scaled)

n_clusters = len(set(county['dbscan_cluster'])) - (1 if -1 in county['dbscan_cluster'].values else 0)
n_noise    = (county['dbscan_cluster'] == -1).sum()
print(f'DBSCAN Results:')
print(f'  Clusters found : {n_clusters}')
print(f'  Noise points   : {n_noise}')
print()
print('Note: DBSCAN classified most counties as one large cluster due to the')
print('relatively uniform spatial density of NY county data. This makes DBSCAN')
print('less suitable here — K-Means and Hierarchical are better choices.')

## 10. Algorithm Comparison & Best Model Selection <a id='10-comparison'></a>

In [ ]:
# ── Summary table ────────────────────────────────────────────────────────
comparison = pd.DataFrame({
    'Algorithm'         : ['K-Means (k=4)', 'Hierarchical (k=4)', 'DBSCAN'],
    'Silhouette ↑'      : [round(km_sil,4), round(hc_sil,4), 'N/A (1 cluster)'],
    'Davies-Bouldin ↓'  : [round(km_db,4),  round(hc_db,4),  'N/A'],
    'Calinski-Harabasz ↑': [round(km_ch,2), round(hc_ch,2),   'N/A'],
    'Interpretability'  : ['High ✅', 'High ✅', 'Low ❌'],
    'Suitable'          : ['✅ Best', '✅ Good', '❌ Poor fit'],
})
print(comparison.to_string(index=False))
print()
print('🏆 WINNER: K-Means (k=4)')
print('   → Best Silhouette Score (0.4311)')
print('   → Lowest Davies-Bouldin Score (0.5317)')
print('   → Highest Calinski-Harabasz Score (60.54)')
print('   → Produces 4 clear, actionable business zones')

## 11. Cluster Profiling & Insights <a id='11-profiling'></a>

In [ ]:
# ── Zone summary ─────────────────────────────────────────────────────────
profile = county.groupby(['cluster','zone_label']).agg(
    num_counties       = ('County','count'),
    total_energy_gwh   = ('total_energy_gwh','sum'),
    avg_energy_gwh     = ('total_energy_gwh','mean'),
    total_projects     = ('total_projects','sum'),
    avg_system_size_kw = ('avg_system_size_kw','mean'),
    avg_developers     = ('unique_developers','mean'),
).round(2).reset_index().drop('cluster',axis=1)
print(profile.to_string(index=False))
print()
print('County assignments per zone:')
for _, row in county.sort_values('cluster').groupby('zone_label'):
    pass
for zone in sorted(county['zone_label'].unique()):
    counties = sorted(county[county['zone_label']==zone]['County'].tolist())
    print(f'\n  {zone} ({len(counties)} counties):')
    print('  ', ', '.join(counties))

In [ ]:
# ── Feature heatmap ───────────────────────────────────────────────────────
FEATURES = ['total_energy_kwh','avg_energy_kwh','total_projects',
            'avg_system_size_kw','total_capacity_kw','energy_per_project','capacity_per_project']

profile_vals = county.groupby('cluster')[FEATURES].mean()
profile_norm = (profile_vals - profile_vals.min()) / (profile_vals.max() - profile_vals.min())
profile_norm.index = [ZONE_NAMES[i] for i in profile_norm.index]
profile_norm.columns = ['Total Energy','Avg Energy/Project','Total Projects',
                         'Avg System Size','Total Capacity','Energy/Project','Capacity/Project']

fig, ax = plt.subplots(figsize=(12,5))
sns.heatmap(profile_norm, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label':'Normalized Score (0-1)'})
ax.set_title('Cluster Zone Feature Heatmap (Normalized 0-1)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 12. Interactive Map Visualization <a id='12-map'></a>
> Uses Plotly's choropleth map with New York State county GeoJSON data for an interactive zone visualization.

In [ ]:
import plotly.express as px
import json, urllib.request

# Load NY counties GeoJSON
url = 'https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json'
with urllib.request.urlopen(url) as response:
    counties_geo = json.load(response)

# NY county FIPS codes mapping
ny_fips = {
    'Albany':'36001','Allegany':'36003','Bronx':'36005','Broome':'36007',
    'Cattaraugus':'36009','Cayuga':'36011','Chautauqua':'36013','Chemung':'36015',
    'Chenango':'36017','Clinton':'36019','Columbia':'36021','Cortland':'36023',
    'Delaware':'36025','Dutchess':'36027','Erie':'36029','Essex':'36031',
    'Franklin':'36033','Fulton':'36035','Genesee':'36037','Greene':'36039',
    'Hamilton':'36041','Herkimer':'36043','Jefferson':'36045','Kings':'36047',
    'Lewis':'36049','Livingston':'36051','Madison':'36053','Monroe':'36055',
    'Montgomery':'36057','Nassau':'36059','New York':'36061','Niagara':'36063',
    'Oneida':'36065','Onondaga':'36067','Ontario':'36069','Orange':'36071',
    'Orleans':'36073','Oswego':'36075','Otsego':'36077','Putnam':'36079',
    'Queens':'36081','Rensselaer':'36083','Richmond':'36085','Rockland':'36087',
    'St Lawrence':'36089','Saratoga':'36091','Schenectady':'36093','Schoharie':'36095',
    'Schuyler':'36097','Seneca':'36099','Steuben':'36101','Suffolk':'36103',
    'Sullivan':'36105','Tioga':'36107','Tompkins':'36109','Ulster':'36111',
    'Warren':'36113','Washington':'36115','Wayne':'36117','Westchester':'36119',
    'Wyoming':'36121','Yates':'36123'
}

map_df = county.copy()
map_df['fips'] = map_df['County'].map(ny_fips)
map_df['Zone'] = map_df['zone_label']
map_df['Total Energy (GWh)'] = map_df['total_energy_gwh'].round(1)
map_df['Avg System Size (kW)'] = map_df['avg_system_size_kw'].round(1)
map_df = map_df.dropna(subset=['fips'])

ZONE_COLOR_MAP = {
    'Zone A — Premium'    : '#e74c3c',
    'Zone B — High Volume': '#3498db',
    'Zone C — Moderate'   : '#f39c12',
    'Zone D — Emerging'   : '#95a5a6',
}

fig = px.choropleth(
    map_df,
    geojson=counties_geo,
    locations='fips',
    color='Zone',
    color_discrete_map=ZONE_COLOR_MAP,
    scope='usa',
    hover_name='County',
    hover_data={'fips':False, 'Total Energy (GWh)':True, 'total_projects':True,
                'Avg System Size (kW)':True, 'Zone':True},
    title='🌞 New York State — Solar Energy Production Zones',
    labels={'total_projects':'Total Projects'},
)
fig.update_geos(fitbounds='locations', visible=False)
fig.update_layout(
    height=550,
    title_font_size=18,
    legend_title_text='Energy Production Zone',
    margin={'r':0,'t':60,'l':0,'b':0},
)
fig.show()

## 13. Business Recommendations <a id='13-recommendations'></a>

---

### 🔴 Zone A — Premium (39 counties)
> *Albany, Erie, Monroe, Nassau, Orange, Queens, Suffolk, Ulster, Westchester, and 30 more.*

- **Strategy:** Top priority for new client acquisition and large-scale project proposals
- **Action:** Deploy dedicated sales teams; prioritize for utility-scale installations
- **Opportunity:** High project density signals mature solar market — target expansions & upgrades

---

### 🔵 Zone B — High Volume (20 counties)
> *Jefferson, Monroe, Oneida, Oswego, St Lawrence, Steuben, and 14 more.*

- **Strategy:** Focus on larger commercial/industrial projects (high avg system size)
- **Action:** Partner with local utilities and developers; feasibility studies for utility-scale
- **Opportunity:** Fewer projects but large average size — ideal for B2B enterprise clients

---

### 🟡 Zone C — Moderate (1 county: Suffolk)
> *Suffolk stands alone as the highest-volume county in NYS.*

- **Strategy:** Suffolk is a market of its own — highest number of projects statewide (51,712)
- **Action:** Invest in dedicated Suffolk operations, community solar, and residential programs
- **Opportunity:** Massive residential solar base — focus on storage, upgrades & NEM programs

---

### ⚫ Zone D — Emerging (2 counties: Allegany, Lewis)
> *Low production, small projects, limited developers.*

- **Strategy:** Long-term growth markets; low competition
- **Action:** Conduct feasibility pilots; engage local municipalities for solar incentives
- **Opportunity:** Early mover advantage — establish presence before competition grows

---

### 📌 Cross-Zone Recommendations
1. **Resource Allocation:** Concentrate 70% of sales & marketing budget on Zone A counties
2. **Product Mix:** Zone B suits large commercial; Zone A & C suit residential + community solar
3. **Developer Partnerships:** Counties with high `unique_developers` signal competitive markets — differentiate on service quality
4. **Data Enrichment:** Add irradiance (GHI/DNI) and rooftop area data for more precise clustering in future iterations
5. **Temporal Analysis:** Track how counties migrate between zones year-over-year as adoption grows


## Final Step — Export Results

In [ ]:
# Save cluster assignments
output_cols = ['County','zone_label','cluster','total_projects','total_energy_gwh',
               'avg_energy_kwh','avg_system_size_kw','total_capacity_kw',
               'energy_per_project','unique_developers','unique_utilities']
county[output_cols].sort_values('total_energy_gwh', ascending=False).to_csv(
    'county_solar_clusters.csv', index=False)
print('✅ Results saved to: county_solar_clusters.csv')
print(f'   {len(county)} counties clustered into 4 energy production zones')
county[output_cols].sort_values('total_energy_gwh', ascending=False).head(10)